In [3]:
import ase
import numpy as np
import pyscf
import time

In [87]:
from pyscf import gto, dft
from pyscf.scf import hf
hf.MUTE_CHKFILE = True

mol = gto.M(atom='O  0  0  0.1184; H  0,  0.7532, -0.4735; H 0,  -0.7532, -0.4735 ', basis='ccpvtz')
mf = dft.RKS(mol)
mf.chkfile=False
mf.xc = 'pbe'
mf.kernel()
g = mf.nuc_grad_method()
g.kernel()

data = np.load('datasets/h2o_dynamic_centered.npy', allow_pickle=True).item()

atom_types = data['atom_types']
mfs = []
mols = []
forces = []
print(len(data['positions']))
for i in range(len(data['positions'])):
    print('calc', i)
    start = time.time()
    pos = data['positions'][i]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis='ccpvtz')
    #print(mol.pack())
    mols.append(mol)
    mf = dft.RKS(mol, conv_tol_grad=1e-9)
    mf.chkfile=False
    mf.xc = 'pbe'
    mf.kernel()
    g = mf.nuc_grad_method()
    mfs.append(mf)
    forces.append(g.grad())
    print('elapsed', time.time() - start)

converged SCF energy = -76.3728513726658
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0000000000     0.0000000000    -0.0133003175
1 H    -0.0000000000    -0.0093879317     0.0066522076
2 H    -0.0000000000     0.0093879317     0.0066522076
----------------------------------------------
4999
calc 0


TypeError: RKS() got an unexpected keyword argument 'conv_tol_grad'

In [77]:
results = []
for i in range(len(mfs)):
    #print(mfs[i].mo_coeff)
    res = mols[i].pack()
    res['mo_coeff'] = mfs[i].mo_coeff
    res['mo_occ'] = mfs[i].mo_occ
    res['energy'] = mfs[i].e_tot
    res['forces'] = forces[i]
    results.append(res)

np.save('datasets/h2o_dynamic_pyscf_dft_f.npy', results)

In [6]:
data_scf = np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)
print(data_scf)
print(len(data_scf))
for d in data_scf:
    print('energy', d['energy'])
    print('forces', d['forces'])

[{'atom': [('O', array([0., 0., 0.], dtype=float32)), ('H', array([-0.766755, -0.276792, -0.656064], dtype=float32)), ('H', array([ 0.229261  ,  0.68619204, -0.40823698], dtype=float32))], 'unit': 'angstrom', 'basis': 'ccpvtz', 'charge': 0, 'spin': 0, 'symmetry': False, 'nucmod': {}, 'nucprop': {}, 'ecp': {}, '_nelectron': None, 'verbose': 3, 'mo_coeff': array([[ 9.72865474e-01, -2.55768283e-02, -4.68619906e-03, ...,
         2.00115700e-01, -2.98432615e-01,  4.02930324e+00],
       [ 1.12084825e-03,  4.64890376e-01, -6.00411786e-02, ...,
         5.19931808e-01, -2.53877003e-01,  6.44933160e+00],
       [ 5.22986030e-02,  1.36445521e-01, -1.50737003e-02, ...,
         4.39437401e-01,  5.99159431e-01, -2.35378392e+00],
       ...,
       [ 2.46238981e-04, -1.32798175e-03,  4.67786866e-04, ...,
        -5.77243459e-02,  2.58679227e-01,  8.42393549e-02],
       [ 5.26734168e-04, -1.50351418e-03, -9.06727772e-04, ...,
         6.42498950e-02,  1.19801753e-01,  2.11390530e-01],
       [ 1.

energy -76.34851743821102
forces [[ 0.02440318  0.10724814 -0.02287954]
 [ 0.02080468 -0.0788297  -0.02107555]
 [-0.04521046 -0.02841365  0.0439573 ]]
energy -76.36519813724888
forces [[ 0.02125848 -0.06058152 -0.03449777]
 [-0.01551877  0.04028109  0.01389823]
 [-0.00573736  0.02029642  0.02059533]]
energy -76.36814177906503
forces [[-0.00090779  0.0174924  -0.03559413]
 [ 0.0001445   0.01079134 -0.00881777]
 [ 0.00076121 -0.02828893  0.04441257]]
energy -76.37258382584933
forces [[-0.01195112 -0.01156143  0.0044315 ]
 [ 0.0029582   0.0041807  -0.00484578]
 [ 0.0089897   0.00737838  0.00041428]]
energy -76.36372817215891
forces [[ 0.04046048  0.08537594  0.05439287]
 [-0.01401339  0.00062437  0.0201566 ]
 [-0.02644458 -0.08599844 -0.07454585]]
energy -76.34268049689331
forces [[ 0.03530317 -0.09372522 -0.03905932]
 [-0.02719371  0.0287408   0.02941301]
 [-0.00811541  0.06497829  0.0096473 ]]
energy -76.34089187909659
forces [[-0.01206242  0.0604228  -0.00802988]
 [-0.00246677  0.03353

 [ 0.01897676  0.02174884 -0.02164759]]
energy -76.33942235132145
forces [[ 0.0028869   0.02228886 -0.04425603]
 [ 0.01156584  0.01070891  0.04476867]
 [-0.01445116 -0.0329961  -0.00051509]]
energy -76.29612055303913
forces [[ 0.07989942  0.13294629  0.46777311]
 [-0.07574239 -0.08763888 -0.45753862]
 [-0.00415412 -0.0453086  -0.01023997]]
energy -76.34111488183913
forces [[-0.00271174 -0.07688794 -0.05489201]
 [-0.03382253  0.02078351  0.03909114]
 [ 0.03653642  0.05610921  0.01580169]]
energy -76.36200422342553
forces [[-0.008387   -0.01819793 -0.01895115]
 [-0.00051419  0.04161269  0.00078167]
 [ 0.00890446 -0.02341663  0.0181708 ]]
energy -76.3591245871762
forces [[-0.05502644 -0.08891521 -0.00385271]
 [ 0.04606759  0.02456704  0.04278086]
 [ 0.00896005  0.06434575 -0.03893182]]
energy -76.36483046040907
forces [[ 0.02500036  0.02638374 -0.00308249]
 [ 0.01114275  0.02427573 -0.00334458]
 [-0.03614391 -0.05065992  0.00642395]]
energy -76.3638703076915
forces [[ 0.04810637 -0.043996

energy -76.35107167141564
forces [[-0.00977445  0.00567137 -0.020108  ]
 [ 0.07120589  0.02501639 -0.00070383]
 [-0.06143052 -0.03068929  0.02081183]]
energy -76.35521945528531
forces [[-0.03856733 -0.00806155  0.01851587]
 [ 0.08034691  0.0159664  -0.0180366 ]
 [-0.04178371 -0.00790831 -0.00047987]]
energy -76.36754462669785
forces [[-0.00135337  0.00709847  0.0205043 ]
 [ 0.00872274 -0.0097176  -0.04557584]
 [-0.00737442  0.00262273  0.02507038]]
energy -76.360494759364
forces [[ 0.00172602  0.00571556  0.01530323]
 [ 0.02158854  0.04567613  0.00775191]
 [-0.02331903 -0.0513907  -0.02305965]]
energy -76.34166110984742
forces [[ 0.07969351 -0.22104595 -0.11639817]
 [-0.02657751  0.01617055  0.01139426]
 [-0.0531157   0.20487641  0.10500511]]
energy -76.362240673491
forces [[-0.01621548 -0.0438216  -0.03072502]
 [ 0.02531692  0.06841391  0.02233631]
 [-0.00909986 -0.02459093  0.00839142]]
energy -76.34497806540085
forces [[ 0.0333677  -0.05921407 -0.10885493]
 [ 0.00307586 -0.00168501 

 [-0.00548661  0.02391997 -0.01772516]]
energy -76.345881603757
forces [[-0.05602952  0.01283024  0.04578627]
 [-0.02245677  0.00543959  0.01851192]
 [ 0.07847538 -0.01826965 -0.06429546]]
energy -76.3657329038311
forces [[ 0.01214486  0.05368153  0.04919255]
 [ 0.00607213 -0.03004673 -0.03809569]
 [-0.01821632 -0.023631   -0.01109654]]
energy -76.36677630310433
forces [[-0.00186509 -0.09021948  0.00951238]
 [-0.01670462  0.03966649  0.02123015]
 [ 0.01856919  0.05055428 -0.03074209]]
energy -76.34117115401139
forces [[ 0.05969306  0.0857657   0.01042044]
 [-0.04935668 -0.09136281 -0.03264517]
 [-0.01033313  0.00559691  0.02222437]]
energy -76.32049117724483
forces [[-0.24513084  0.01875573  0.0421236 ]
 [ 0.27751469  0.02509476 -0.01292246]
 [-0.03238726 -0.04385181 -0.02919826]]
energy -76.34446713754554
forces [[ 0.06069703 -0.04851144  0.07911376]
 [-0.07470305  0.02883795 -0.04020621]
 [ 0.01400255  0.01967531 -0.03891321]]
energy -76.37002445437957
forces [[ 0.00781012 -0.0129354

...

forces [[-0.00314566  0.02093584  0.05639324]
 [ 0.05959216  0.02577567  0.00077178]
 [-0.05644488 -0.0467065  -0.05716298]]
energy -76.35332252411976
forces [[-0.0565539  -0.04061969 -0.08453066]
 [ 0.01606136  0.02313406  0.03418099]
 [ 0.04048861  0.01748636  0.05034546]]
energy -76.37029761331937
forces [[ 0.03834162 -0.02319702  0.01183368]
 [-0.01470722  0.0001462  -0.00122095]
 [-0.02363522  0.02305328 -0.01061023]]
energy -76.36456810769555
forces [[-0.01592933  0.01232474 -0.01642241]
 [ 0.03502913  0.0191501   0.03996727]
 [-0.0191009  -0.03147152 -0.02353968]]
energy -76.36507909849925
forces [[ 0.00877925 -0.04043699  0.02481794]
 [-0.01419234 -0.01909874 -0.03685833]
 [ 0.00541285  0.05953484  0.01203651]]
energy -76.37108449730438
forces [[-0.00660021 -0.02349559  0.0310615 ]
 [ 0.01694935  0.02221399 -0.02645626]
 [-0.01035053  0.00128244 -0.00460145]]
energy -76.3564693028828
forces [[-0.02867175  0.00652329 -0.07787133]
 [-0.01833271  0.02653136  0.01168985]
 [ 0.04701

energy -76.35886140322461
forces [[-3.73085965e-03  6.65521680e-02  6.32660916e-02]
 [ 2.56862626e-05 -3.22599306e-02 -3.56892625e-02]
 [ 3.70852955e-03 -3.42941476e-02 -2.75779541e-02]]
energy -76.31450102344633
forces [[-0.02043151  0.13719071 -0.00141765]
 [-0.01892361 -0.11679052 -0.00529465]
 [ 0.03935682 -0.02039661  0.0067081 ]]
energy -76.36342671552222
forces [[-0.00167122  0.03552347  0.06050687]
 [-0.00071132 -0.02182202 -0.03136117]
 [ 0.00238423 -0.01370211 -0.02914929]]
energy -76.36972936518127
forces [[ 0.00175421 -0.03368401  0.04303099]
 [ 0.00147031  0.00733158 -0.04249309]
 [-0.00322434  0.02635232 -0.00053602]]
energy -76.35605391835868
forces [[ 0.01738629  0.0915455   0.04670827]
 [-0.03123025 -0.01864704 -0.03185016]
 [ 0.01385006 -0.07289601 -0.01485734]]
energy -76.36711476466692
forces [[-0.01621589  0.01768692 -0.02503535]
 [-0.01035602  0.01722856 -0.00954376]
 [ 0.0265719  -0.03491716  0.03457667]]
energy -76.33418323191782
forces [[-0.13258297  0.09623801

forces [[ 0.13564851 -0.0766055   0.07644899]
 [ 0.03187505 -0.03082606  0.00408853]
 [-0.16752126  0.10743279 -0.08053243]]
energy -76.33883297585331
forces [[-0.08899148 -0.06730825  0.0666598 ]
 [ 0.00823698  0.04463228 -0.020833  ]
 [ 0.08075376  0.02268283 -0.04582578]]
energy -76.36064278871254
forces [[-0.04918252 -0.05591422  0.01033277]
 [-0.01882468  0.0238093  -0.02722921]
 [ 0.0680125   0.03210425  0.01689676]]
energy -76.33499694036692
forces [[-0.03055751  0.02816435  0.02412236]
 [ 0.01452515  0.03055442  0.03933814]
 [ 0.01603483 -0.05871839 -0.06346035]]
energy -76.35680801228797
forces [[ 0.08108276 -0.03112901  0.01368365]
 [-0.05835535  0.00900996 -0.01172399]
 [-0.02273052  0.02212075 -0.00195525]]
energy -76.34750449417783
forces [[-0.06279605 -0.00415999 -0.0895479 ]
 [ 0.01481923 -0.00275185  0.01285605]
 [ 0.04797258  0.00691089  0.07669205]]
energy -76.36740292671357
forces [[-0.00453833 -0.04850532  0.04690982]
 [ 0.00394246  0.05334294 -0.06170297]
 [ 0.0005

forces [[-0.07672329 -0.04885413 -0.01264242]
 [ 0.01493488  0.01238754 -0.00443303]
 [ 0.0617945   0.03646542  0.01707806]]
energy -76.33956762949602
forces [[ 0.14497375  0.12505943 -0.10332923]
 [-0.02152608 -0.07502855  0.03442218]
 [-0.12344991 -0.05003255  0.06890816]]
energy -76.36177784558991
forces [[ 0.12502328  0.0540957   0.02666968]
 [-0.01175767 -0.02939313 -0.02993442]
 [-0.11326427 -0.0247015   0.00327093]]
energy -76.37269359993189
forces [[ 0.00585714  0.00555699 -0.01708066]
 [ 0.00182259 -0.00519363  0.00054727]
 [-0.00767878 -0.0003603   0.01653384]]
energy -76.35081112711175
forces [[-0.00569749  0.20747912 -0.01525021]
 [ 0.03922936 -0.03722069  0.00915041]
 [-0.03353322 -0.17026065  0.00609849]]
energy -76.34795486084971
forces [[-0.09242747 -0.00366006  0.07674282]
 [ 0.00945971  0.01070307 -0.06315148]
 [ 0.08296876 -0.00704183 -0.01359118]]
energy -76.36275006233768
forces [[ 0.12705465 -0.03024485  0.02029013]
 [-0.03226005 -0.00896749  0.02034132]
 [-0.0947

 [-0.01835064  0.02429073  0.03475234]]
energy -76.33364139557439
forces [[ 0.23449752  0.0168128  -0.00804534]
 [-0.08784025  0.03632446 -0.03184915]
 [-0.14665485 -0.05314164  0.03989187]]
energy -76.36754863901909
forces [[ 0.01406543 -0.00672539 -0.0197299 ]
 [ 0.00712412 -0.01370448  0.01840344]
 [-0.02119413  0.02042989  0.00133082]]
energy -76.36851220363239
forces [[ 2.92708650e-05  3.47579002e-02 -1.83072106e-02]
 [-1.96439563e-02 -2.52380944e-02  3.91370920e-02]
 [ 1.96101098e-02 -9.52259305e-03 -2.08277606e-02]]
energy -76.35165633504907
forces [[ 0.0204705   0.0179661   0.01875311]
 [-0.03277641  0.00905485 -0.07514796]
 [ 0.01230244 -0.02702579  0.05639456]]
energy -76.3536005963898
forces [[-0.01407233 -0.07143636 -0.00915824]
 [ 0.00870835 -0.03793583 -0.00834497]
 [ 0.00536479  0.10937162  0.01750145]]
energy -76.3660541827627
forces [[ 0.05069845 -0.02005627  0.0279911 ]
 [-0.03379706  0.0104357  -0.01539798]
 [-0.01690416  0.0096216  -0.01259268]]
energy -76.359379160

In [7]:
for d in data_scf:
    d['forces'] = d['forces'] * 0.529177


np.save('datasets/h2o_dynamic_pyscf_dft_f.npy', data_scf)

In [15]:
new_data = []
data_scf = data_scf = np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)
for d in data_scf:
    new_d = []
    mo_coeff = d.pop('mo_coeff')
    mo_occ = d.pop('mo_occ')
    en = d.pop('energy')
    f = d.pop('forces')
    new_d.append(d)
    new_d.append({'mo_coeff': mo_coeff, 'mo_occ': mo_occ, 'energy': en, 'forces': f})
    new_data.append(new_d)

np.save('datasets/h2o_dynamic_pyscf_dft_f_en.npy', new_data)

In [81]:
results = {'E': [], 'F': [], 'R': [], 'z': np.array([8, 1, 1])}
for i in range(len(mfs)):
    #print(mfs[i].mo_coeff)
    res = mols[i].pack()
    pos = []
    for a in res['atom']:
        pos.append(a[1])
    pos = np.array(pos)
    print('pos.shape', pos.shape)
    results['R'].append(pos)
    results['E'].append(mfs[i].e_tot)
    results['F'].append(forces[i])
    
results['R'] = np.array(results['R'])
results['E'] = np.array(results['E'])
results['F'] = np.array(results['F'])

np.savez('datasets/water_pyscf_dft_f', **results)

pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 

pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 

pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 

pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 

In [96]:
results = {'energy': [], 'forces': [], 'positions': [],
           'atom_numbers': [8, 1, 1], 'atom_types': ['O', 'H', 'H'],
          'mo_coeff': [], 'mo_occ': []}
for d in data_scf:
    #print(mfs[i].mo_coeff)
    pos = []
    for a in d['atom']:
        pos.append(a[1])
    pos = np.array(pos)
    print('pos.shape', pos.shape)
    results['positions'].append(pos)
    results['energies'].append(d['energy'])
    results['forces'].append(d['forces'])
    results['mo_coeff'].append(d['mo_coeff'])
    results['mo_occ'].append(d['mo_occ'])

for key in results.keys():
    results[key] = np.array(results[key])
    
np.savez('datasets/h2o_dynamic_pyscf_dft_f', **results)

pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 

pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 

pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 

pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 3)
pos.shape (3, 

In [16]:
print(np.load('datasets/h2o_dynamic_pyscf_dft.npy', allow_pickle=True)[0])
print(np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)[0])
print(np.load('datasets/h2o_dynamic_pyscf_dft_f_en.npy', allow_pickle=True)[0])

[{'atom': [['O', array([0., 0., 0.], dtype=float32)], ['H', array([-0.766755, -0.276792, -0.656064], dtype=float32)], ['H', array([ 0.229261  ,  0.68619204, -0.40823698], dtype=float32)]], 'unit': 'angstrom', 'basis': 'augccpvdz', 'charge': 0, 'spin': 0, 'symmetry': False, 'nucmod': {}, 'nucprop': {}, 'ecp': {}, '_nelectron': None, 'verbose': 3}
 {'mo_coeff': array([[ 1.00528933e+00, -6.64707489e-03, -1.89939307e-04, ...,
        -4.90113168e-07, -3.18101324e-01,  2.35683627e-02],
       [ 1.20463894e-02,  4.38377124e-01, -4.52471295e-02, ...,
        -9.45287200e-07, -5.49921044e-01, -6.03560487e-03],
       [-1.87840997e-02,  3.07188336e-01, -1.24077372e-01, ...,
         3.26228082e-06,  3.11562467e+00, -2.88710339e+00],
       ...,
       [-1.21801881e-03, -2.52780531e-03, -2.67088422e-03, ...,
         3.34710287e-01,  2.78715926e-01, -4.41107601e-01],
       [-3.81720541e-03,  1.49184372e-03, -1.40695034e-02, ...,
        -2.75427177e-01,  4.28217795e-01, -1.07297478e+00],
      